In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/train.json
/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/questions_clean.json
/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/law.json


In [2]:
# Cài đặt các phiên bản thư viện đã được kiểm chứng an toàn cho Kaggle
!pip install -q "sentence-transformers<3.3.0" "transformers<4.48.0" pandas torch

# Khởi tạo môi trường sạch cho CUDA
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Cài đặt thư viện hoàn tất! Sẵn sàng xử lý dữ liệu.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 84.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incomp

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ==========================================
# 1. TẢI DỮ LIỆU ĐẦU VÀO & DỰNG TỪ ĐIỂN TRA CỨU
# ==========================================
print("Đang đọc dữ liệu luật và tập câu hỏi...")
with open('/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/law.json', 'r', encoding='utf-8') as f:
    laws = json.load(f)

with open('/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/questions_clean.json', 'r', encoding='utf-8') as f:
    clean_questions_t1 = json.load(f)

law_dict = {item['chunk_id']: item['text_to_embed'] for item in laws}

final_perfect_clean = []
logical_noisy_questions = []

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH QWEN-1.5B
# ==========================================
print("Đang tải mô hình Qwen2.5-1.5B-Instruct...")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto"
)

# ==========================================
# 3. VÒNG LẶP THẨM ĐỊNH ĐÃ ĐƯỢC NỚI LỎNG LOGIC
# ==========================================
print("LLM đang tiến hành kiểm tra chéo logic với cấu hình nới lỏng...")

# Giữ nguyên [:200] để test nhanh. Khi bấm Save Version toàn bộ, hãy XÓA [:200] đi nhé!
for q in tqdm(clean_questions_t1): 
    assigned_id = q['article_id'][0]
    
    if assigned_id not in law_dict:
        continue
        
    law_content = law_dict[assigned_id]
    
    # THAY ĐỔI CỐT LÕI: Prompt mới thân thiện hơn, tập trung vào việc "Đoạn luật có liên quan để giải quyết không"
    prompt = f"""Bạn là một trợ lý phân tích dữ liệu pháp luật Việt Nam. Hãy đọc kỹ đoạn văn bản luật và câu hỏi tình huống dưới đây:

[Văn bản Luật Căn Cứ]:
{law_content}

[Câu Hỏi Tình Huống]:
{q['question']}

[Nhiệm vụ]: Hãy cho biết văn bản Luật trên có phải là căn cứ chính xác, hoặc có chứa thông tin liên quan trực tiếp giúp giải quyết, trả lời cho tình huống trong Câu hỏi hay không?
- Hãy trả lời 'YES' nếu đoạn luật trên nói về cùng một chủ đề và cung cấp quy định pháp lý để giải quyết câu hỏi.
- Chỉ trả lời 'NO' nếu đoạn luật và câu hỏi hoàn toàn lệch pha, thuộc hai chủ đề khác nhau, hoặc không có một chút liên quan nào.

Chỉ trả ra đúng một từ duy nhất: 'YES' hoặc 'NO'. Không giải thích gì thêm."""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=2, 
            temperature=0.3, # Tăng nhẹ temperature để mô hình linh hoạt hơn, không bị cứng nhắc
            do_sample=True,   # Kích hoạt sampling nhẹ để tránh bẫy tự động "Say NO"
            top_k=5          # Giới hạn không gian token sinh
        )
    
    response = tokenizer.batch_decode(generated_ids[:, model_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip().upper()

    if "YES" in response:
        final_perfect_clean.append(q)
    else:
        q['logical_error_diagnostic'] = "LLM nghi ngờ độ lệch logic chủ đề."
        logical_noisy_questions.append(q)

# ==========================================
# 4. XUẤT FILE BÁO CÁO KẾT QUẢ
# ==========================================
print(f"\n🎯 KẾT QUẢ SAU KHI NỚI LỎNG PROMPT:")
print(f"  - Số câu hợp lệ (YES): {len(final_perfect_clean)}")
print(f"  - Số câu nghi vấn (NO) : {len(logical_noisy_questions)}")

with open('llm_questions_perfect_clean.json', 'w', encoding='utf-8') as f:
    json.dump(final_perfect_clean, f, ensure_ascii=False, indent=4)
    
with open('llm_questions_logical_noisy.json', 'w', encoding='utf-8') as f:
    json.dump(logical_noisy_questions, f, ensure_ascii=False, indent=4)

Đang đọc dữ liệu luật và tập câu hỏi...
Đang tải mô hình Qwen2.5-1.5B-Instruct...
LLM đang tiến hành kiểm tra chéo logic với cấu hình nới lỏng...


100%|██████████| 4897/4897 [48:35<00:00,  1.68it/s] 


🎯 KẾT QUẢ SAU KHI NỚI LỎNG PROMPT:
  - Số câu hợp lệ (YES): 3778
  - Số câu nghi vấn (NO) : 1119
